# Late Interaction Retrieval - 02: Token embeddings and MaxSim

> **MLCourse - Agentic AI - Advanced RAG - Module 15**

Notebook 01 previewed MaxSim in four lines and waved at the details. This
notebook builds it properly and explains every one of those details, because
each of them is a real bug you can ship.

### What you will learn

1. How to pull per-token embeddings out of a local model.
2. Why the vectors must be **length-normalised** first.
3. The MaxSim formula, written out and then implemented.
4. Why it is **asymmetric** (max over documents, sum over queries) - and what
   breaks if you do it the other way round.
5. How to handle **padding** when you batch, which is where the real bugs are.

Still no LLM calls, still no API key.

### Setup


In [ ]:
import numpy as np
import torch
from sentence_transformers import SentenceTransformer

MODEL = SentenceTransformer("all-MiniLM-L6-v2")
TOKENIZER = MODEL.tokenizer
DIM = MODEL.get_embedding_dimension()

print(f"model: all-MiniLM-L6-v2, {DIM} dims per token")


def tokens_of(text):
    """The human-readable token strings, so our matrices have labels."""
    return TOKENIZER.convert_ids_to_tokens(TOKENIZER(text)["input_ids"])


QUERY = "what is the capital city of France"
DOC_GOOD = "Paris is the capital and most populous city of France, on the river Seine."
DOC_TRAP = "French cuisine is celebrated worldwide; Lyon is often called its capital."

print(f"\nquery    : {QUERY}")
print(f"good doc : {DOC_GOOD}")
print(f"trap doc : {DOC_TRAP}")
print("\nThe trap doc contains 'capital' and 'French' but answers nothing.")


### 1. Getting token embeddings

`sentence-transformers` normally returns a *pooled* vector: the transformer
produces one vector per token, and a pooling layer averages them into one.
`output_value="token_embeddings"` returns the layer **before** pooling.

That is the whole trick. There is no special model required to *obtain* token
embeddings - every transformer produces them, and pooling throws them away.

### Pulling the pre-pooling token matrix


In [ ]:
q_emb = MODEL.encode(QUERY, output_value="token_embeddings")
d_emb = MODEL.encode(DOC_GOOD, output_value="token_embeddings")

print(f"query tokens : {len(tokens_of(QUERY))} -> matrix {tuple(q_emb.shape)}")
print(f"doc tokens   : {len(tokens_of(DOC_GOOD))} -> matrix {tuple(d_emb.shape)}")
print(f"type: {type(q_emb).__name__}  (a torch tensor, not a numpy array)")

# Compare against the pooled vector to make the loss concrete:
pooled = MODEL.encode(DOC_GOOD)
print(f"\npooled doc vector: {pooled.shape} = {q_emb.shape[1]} numbers")
print(f"token doc matrix : {tuple(d_emb.shape)} = {d_emb.shape[0] * d_emb.shape[1]} numbers")
print(f"ratio: {d_emb.shape[0]}x more numbers to store for this one passage")


### 2. Why normalisation is not optional

MaxSim is defined over **cosine similarity**, and cosine similarity is the dot
product *of unit vectors*. If you skip the normalisation and dot the raw
vectors, you are computing an unnormalised dot product, whose magnitude
depends on how long each token's vector happens to be.

That is not a cosmetic difference. Token vector norms vary a lot - common
function words and structural tokens often have larger norms than content
words. Skip normalisation and a passage can win on `[SEP]` having a big
vector. Let's measure that rather than assert it.

### Token vector norms vary, so normalisation changes the answer


In [ ]:
norms = d_emb.norm(dim=-1).numpy()
labels = tokens_of(DOC_GOOD)

print("Raw L2 norm of each token vector in the good doc:")
for tok, n in zip(labels, norms):
    bar = "#" * int(n * 3)
    print(f"  {tok:<10} {n:6.3f}  {bar}")

print(f"\nsmallest {norms.min():.3f}   largest {norms.max():.3f}"
      f"   ratio {norms.max() / norms.min():.2f}x")
print("\nA >1x spread means an unnormalised dot product is partly a")
print("popularity contest between token norms, not a similarity measure.")


### The normalisation helper we will use everywhere


In [ ]:
def normalize(mat):
    """L2-normalise every ROW so that a dot product equals cosine similarity.

    Takes a torch tensor or numpy array of shape (n_tokens, dim);
    returns a float32 numpy array of the same shape.
    """
    arr = mat.detach().cpu().numpy() if isinstance(mat, torch.Tensor) else np.asarray(mat)
    arr = arr.astype(np.float32)
    # keepdims so the division broadcasts row-wise; the tiny epsilon guards
    # against a zero-length vector producing NaN.
    return arr / (np.linalg.norm(arr, axis=-1, keepdims=True) + 1e-12)


q_norm = normalize(q_emb)
d_norm = normalize(d_emb)

print("after normalisation, every row has length 1.0:")
print(" ", np.round(np.linalg.norm(q_norm, axis=-1), 6))


### 3. The MaxSim formula

Write the query as a set of $n$ token vectors $q_1 \dots q_n$ and the document
as $m$ token vectors $d_1 \dots d_m$. The late-interaction score is:

$$\text{MaxSim}(Q, D) \;=\; \sum_{i=1}^{n} \; \max_{j=1..m} \; q_i \cdot d_j$$

Read it left to right as an instruction:

1. **For each query token** $q_i$ …
2. … look at **every** document token and take the **best** match (`max`) …
3. … and **add up** those bests across the query (`sum`).

In code that is three lines: one matrix multiply, one `max`, one `sum`. That
is genuinely all a late-interaction model does at scoring time - which is why
it is cheap enough to run over a candidate set, unlike a cross-encoder.

### Why max over documents, and sum over queries?

This asymmetry is the design, not an accident, and it encodes two different
claims.

**`max` over the document** says: *a query term is satisfied if it is found
**anywhere** in the passage.* A passage should not be penalised for being long
and containing other material. Averaging over document tokens instead would
punish exactly that, and you would have reinvented the pooling you were trying
to escape.

**`sum` over the query** says: *every query term must be accounted for
separately.* A passage that nails "France" but has nothing for "capital" gets
credit for one term only. This is what makes MaxSim behave a little like a
term-matching system (`BM25`, from `01_hybrid_search`) while still operating
on semantics rather than exact strings.

Flip either one and you break it:

| Variant | What goes wrong |
|---|---|
| `mean` over doc instead of `max` | long passages are penalised; you are back to pooling |
| `max` over query instead of `sum` | one strong term match wins the whole passage; the other query terms stop mattering |
| `max` over query *and* doc | the score becomes "best single word pair anywhere" - nearly useless |

We will measure that last claim rather than assert it.

### MaxSim, implemented


In [ ]:
def maxsim(query_tokens, doc_tokens, return_matrix=False):
    """The late-interaction score between one query and one document.

    query_tokens : (n, dim) L2-normalised query token vectors
    doc_tokens   : (m, dim) L2-normalised document token vectors

    Returns the scalar score, and optionally the (n, m) similarity matrix
    it was computed from -- that matrix is what makes the score explainable.
    """
    # Step 1: every query token against every doc token, in one matmul.
    sim = query_tokens @ doc_tokens.T          # shape (n, m)

    # Step 2: for each query token, its single best doc token.
    best_per_query_token = sim.max(axis=1)     # shape (n,)

    # Step 3: add them up.
    score = float(best_per_query_token.sum())

    return (score, sim) if return_matrix else score


good_score, good_sim = maxsim(q_norm, d_norm, return_matrix=True)
trap_norm = normalize(MODEL.encode(DOC_TRAP, output_value="token_embeddings"))
trap_score, trap_sim = maxsim(q_norm, trap_norm, return_matrix=True)

print(f"MaxSim(query, GOOD doc) = {good_score:.4f}")
print(f"MaxSim(query, TRAP doc) = {trap_score:.4f}")
print(f"margin = {good_score - trap_score:+.4f}"
      f"   ({'good doc wins' if good_score > trap_score else 'TRAP WINS -- bad'})")


### 4. Reading the score apart

The number on its own is no better than a cosine. What makes MaxSim different
is that the number is a **sum of per-query-token contributions**, so you can
print the breakdown and see exactly where one passage beat the other.

### Per-token contribution breakdown: WHY one doc beat the other


In [ ]:
q_labels = tokens_of(QUERY)
good_labels = tokens_of(DOC_GOOD)
trap_labels = tokens_of(DOC_TRAP)

print(f"{'query token':<12} | {'best in GOOD doc':<22} | {'best in TRAP doc':<22} | delta")
print("-" * 78)
total_delta = 0.0
for i, qt in enumerate(q_labels):
    gj, tj = int(good_sim[i].argmax()), int(trap_sim[i].argmax())
    gs, ts = good_sim[i, gj], trap_sim[i, tj]
    total_delta += gs - ts
    print(f"{qt:<12} | {good_labels[gj]:<10} {gs:6.3f}       "
          f"| {trap_labels[tj]:<10} {ts:6.3f}       | {gs - ts:+6.3f}")
print("-" * 78)
print(f"{'TOTAL':<12} | {good_score:>17.3f}     | {trap_score:>17.3f}     | {total_delta:+6.3f}")


### What that table tells you

Scan the `delta` column for the rows that carry the decision. Content tokens
(`capital`, `city`, `france`) are where a real separation should appear;
structural tokens (`[CLS]`, `[SEP]`, `the`, `is`) contribute similar amounts
to both passages and mostly cancel out.

Note the honest wrinkle: the trap document *does* contain the literal token
`capital`, so that row will not separate the two documents much on its own.
The separation has to come from the tokens the trap has no answer for. If the
totals are close, that is a real result about this model, not a bug - and
notebook 03 measures it across a whole corpus rather than one pair.

> **Reminder from notebook 01:** we are applying MaxSim to a bi-encoder's
> token embeddings. A trained ColBERT model learns embeddings *for* this
> operator and separates these cases far more sharply. The mechanism here is
> real; the margins are borrowed.

### 5. Padding - where the real bugs live

Scoring one document at a time, as above, is clear but slow. In practice you
batch: encode 1000 documents at once and store them as a single 3-D array of
shape `(n_docs, max_tokens, dim)`.

To make that a rectangle, short documents get **padded** with zero rows.

Here is the bug: a padded row is a zero vector, so `q · pad = 0`. That sounds
harmless. It is not - the `max` is over *raw similarity*, and cosine
similarity ranges over `[-1, 1]`. A query token whose genuine best match is a
**negative** similarity (the passage has nothing for it) will happily pick a
padding row instead, scoring `0.0` - better than its real best.

The effect is that padding can silently **inflate the scores of short
documents**, precisely for the query tokens the document failed to answer.
That is the worst possible place to leak credit.

The fix is a **mask**: set padded positions to `-inf` (or any value below -1)
before the `max`, so they can never win.

We are going to run this experiment on our corpus and then look honestly at
what it shows - including the possibility that it shows nothing.

### Batched MaxSim, with the padding mask done correctly


In [ ]:
def encode_tokens(texts):
    """Encode a list of texts into per-text (n_tokens, dim) normalised arrays."""
    embs = MODEL.encode(list(texts), output_value="token_embeddings")
    return [normalize(e) for e in embs]


def pad_batch(token_mats):
    """Stack ragged (n_i, dim) matrices into one (N, max_n, dim) array + mask.

    Returns:
      padded : (N, max_n, dim) float32, zero-padded
      mask   : (N, max_n) bool, True where the row is a REAL token
    """
    n_docs = len(token_mats)
    max_n = max(m.shape[0] for m in token_mats)
    padded = np.zeros((n_docs, max_n, DIM), dtype=np.float32)
    mask = np.zeros((n_docs, max_n), dtype=bool)
    for i, m in enumerate(token_mats):
        padded[i, : m.shape[0]] = m
        mask[i, : m.shape[0]] = True
    return padded, mask


def maxsim_batch(query_tokens, padded_docs, doc_mask):
    """Score one query against a whole batch of documents at once.

    query_tokens : (n, dim)
    padded_docs  : (N, m, dim)
    doc_mask     : (N, m) True for real tokens

    Returns (N,) scores.
    """
    # einsum reads: for each doc d, each query token i, each doc token j,
    # sum over the dim axis k. Result: (N, n, m).
    sim = np.einsum("ik,djk->dij", query_tokens, padded_docs)

    # THE IMPORTANT LINE. Without it, padded zero-columns can win the max
    # for any query token whose real best similarity is negative.
    sim = np.where(doc_mask[:, None, :], sim, -np.inf)

    return sim.max(axis=2).sum(axis=1)


CORPUS = [
    "The Eiffel Tower was completed in 1889 and stands on the Champ de Mars in Paris.",
    DOC_GOOD,
    "France is a country in Western Europe with several overseas territories.",
    "The capital of Italy is Rome, a city famous for the Colosseum and the Forum.",
    "Berlin became the capital of a reunified Germany in 1990.",
    "Many capital cities grew up around a river crossing or a defensible hill.",
    DOC_TRAP,
    "The Louvre, in central Paris, is the world's most-visited art museum.",
    "Yes.",   # a deliberately tiny doc, to make the padding effect visible
]

doc_mats = encode_tokens(CORPUS)
padded, mask = pad_batch(doc_mats)
print(f"batched into {padded.shape} ({padded.shape[0]} docs, "
      f"padded to {padded.shape[1]} tokens, {padded.shape[2]} dims)")
print(f"real tokens per doc: {mask.sum(axis=1).tolist()}")
print(f"padding waste: {100 * (1 - mask.sum() / mask.size):.1f}% of the array is padding")


### Proving the padding bug is real, not theoretical


In [ ]:
masked_scores = maxsim_batch(q_norm, padded, mask)

# The buggy version: no mask, so zero-padding rows compete in the max.
sim_unmasked = np.einsum("ik,djk->dij", q_norm, padded)
unmasked_scores = sim_unmasked.max(axis=2).sum(axis=1)

print(f"{'doc':<5} {'tokens':>7} {'correct':>9} {'buggy':>9} {'inflation':>11}")
print("-" * 46)
for i in range(len(CORPUS)):
    infl = unmasked_scores[i] - masked_scores[i]
    flag = "  <== inflated" if infl > 1e-6 else ""
    print(f"D{i:<4} {int(mask[i].sum()):>7} {masked_scores[i]:>9.4f} "
          f"{unmasked_scores[i]:>9.4f} {infl:>+11.4f}{flag}")

print("\nRanking with the CORRECT (masked) scores:")
for rank, i in enumerate(np.argsort(-masked_scores), start=1):
    print(f"  {rank}. [{masked_scores[i]:.4f}] D{i}: {CORPUS[i][:60]}")


### The experiment says: no inflation at all. Why?

Every row of the `inflation` column is `+0.0000`. The buggy version and the
correct version agree exactly.

**That is a real result and we are not going to hide it.** The reason is a
property of this particular model, and it is worth measuring rather than
guessing at: a padding row only wins the `max` if the query token's genuine
best similarity was **below zero**. If every real similarity is positive, the
zero rows never win, and the mask changes nothing.

So the question becomes: how close does this model get to zero?

### Why the mask did nothing here: measure the similarity floor


In [ ]:
# The true (mask-respecting) best similarity for every (doc, query token) pair.
real_best = np.where(mask[:, None, :], sim_unmasked, -np.inf).max(axis=2)

print(f"lowest 'best match' similarity anywhere in this corpus: {real_best.min():.4f}")
print(f"highest                                               : {real_best.max():.4f}")
print()
if real_best.min() > 0:
    print("Every value is POSITIVE, so a zero-valued padding row could never")
    print("win a max. That is why masking changed nothing on this corpus.")
    print()
    print("This is a property of all-MiniLM-L6-v2, not a property of MaxSim:")
    print("its embedding space is anisotropic -- almost all token vectors point")
    print("into a narrow cone, so almost any two of them have positive cosine.")
    print("Models with better-spread embedding spaces DO produce negatives.")
else:
    print("Some best-matches are negative, so padding rows can and do win.")


### The bug, demonstrated on a case that DOES trigger it


In [ ]:
# Since our real model never produces a negative best-match, we construct a
# tiny synthetic example by hand. These are made-up 2-D unit vectors, clearly
# labelled as synthetic -- the point is to show the arithmetic, not to claim
# this happens with MiniLM.

# Two query tokens, pointing along the first two axes of a 3-D space.
q_synth = np.array([[1.0, 0.0, 0.0],
                    [0.0, 1.0, 0.0]], dtype=np.float32)

# Doc A: SHORT (2 real tokens), and it genuinely OPPOSES query token 0 --
# both of its tokens have a negative similarity to it.
doc_a = np.array([[-0.300, 0.900, 0.316],
                  [-0.500, 0.500, 0.707]], dtype=np.float32)

# Doc B: LONGER (3 real tokens), a modest but genuinely positive match to both.
doc_b = np.array([[0.200, 0.500, 0.842],
                  [0.100, 0.600, 0.794],
                  [0.150, 0.550, 0.821]], dtype=np.float32)

padded_s = np.zeros((2, 4, 3), dtype=np.float32)   # both padded out to 4 rows
padded_s[0, :2], padded_s[1, :3] = doc_a, doc_b
mask_s = np.zeros((2, 4), dtype=bool)
mask_s[0, :2], mask_s[1, :3] = True, True

sim_s = np.einsum("ik,djk->dij", q_synth, padded_s)
correct_s = np.where(mask_s[:, None, :], sim_s, -np.inf).max(axis=2).sum(axis=1)
buggy_s = sim_s.max(axis=2).sum(axis=1)

print("SYNTHETIC example (hand-made vectors, not model output)\n")
print("Doc A, query token 0 -- similarities to its two real tokens:",
      np.round(sim_s[0, 0, :2], 3), " <- both NEGATIVE for the first")
print()
print(f"{'doc':<6} {'correct':>9} {'buggy':>9} {'inflation':>11}")
print("-" * 38)
for i, name in enumerate(["A", "B"]):
    print(f"{name:<6} {correct_s[i]:>9.4f} {buggy_s[i]:>9.4f} "
          f"{buggy_s[i] - correct_s[i]:>+11.4f}")

print(f"\ncorrect ranking: {'B beats A' if correct_s[1] > correct_s[0] else 'A beats B'}")
print(f"buggy ranking  : {'B beats A' if buggy_s[1] > buggy_s[0] else 'A beats B'}")
print("\nDoc A picked up free credit from a padding row for the query token")
print("it genuinely could not answer -- exactly the wrong place to leak credit.")


### Interpreting the padding experiment

Two findings, and they matter in different ways.

**On our real corpus, masking changed nothing.** `all-MiniLM-L6-v2` produces
an anisotropic embedding space - its token vectors all point into a fairly
narrow cone, so nearly any two of them have a positive cosine similarity. A
zero-valued padding row is therefore *worse* than every real option and never
wins the `max`. If you only ever use this model, you could ship the unmasked
version and never notice.

**On the synthetic example, masking changed the ranking.** Once a query token
has a genuinely negative best match, the padding row wins, and the document
that deserved credit for *nothing* gets credit for zero - which is more. The
shortest documents carry the most padding rows and are therefore the most
exposed, so the bias runs toward rewarding documents for containing less.

The generalisable lesson is not "padding always inflates scores". It is:

> Whether this bug bites depends on a property of your embedding space that
> you probably have not checked. Mask anyway - it costs one line - and if you
> want to know whether it mattered, measure the similarity floor as we did
> above.

**The rule to remember:** in late interaction, *always mask before the max*.

### Key takeaways

- Token embeddings are just the transformer output **before pooling** -
  `output_value="token_embeddings"`. No special model is needed to get them.
- **Normalise every token vector** before scoring. Token norms vary enough
  that an unnormalised dot product measures the wrong thing.
- **MaxSim = max over document tokens, then sum over query tokens.** The
  asymmetry is deliberate: `max` means "found anywhere in the passage",
  `sum` means "every query term counted separately".
- The score **decomposes per query token**, which makes late interaction
  unusually debuggable.
- When batching, **mask the padding before taking the max**, or short
  documents get silently inflated scores.

**Next:** `03_late_interaction_retrieval.ipynb` - running this over a real
corpus and comparing it against single-vector retrieval on a fixed question
set.